# **BONUS MATERIAL: Access and analyze pre-calculated climate indicators for an ensemble of climate simulations**

This notebook shows how to create additional figures from indicator data generated in `tutorial-indicators.ipynb`.

First re-run **Steps 1 to 7** in `tutorial-indicators.ipynb` using the input settings specified for each example below. Each new combination of inputs will generate a new dataset and will save it as a `.zarr` file in the output folder.

Once the new datasets have been created and saved, run the code provided in each section to generate the figures.

### **Preparation – Imports & configuration**

Run this cell once at the very beginning of the notebook to load the Python packages. No modification needed

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import geopandas as gpd
import hvplot.pandas
import hvplot.xarray
import panel as pn
import xarray as xr
from xscen import ensembles as xens
warnings.simplefilter("ignore")
pn.extension("tabulator")

#### **Example 2: Create a time series of seasonal precipitation for a sub-region under different SSP scenarios**

In Example 2, the goal is to:
  and
1. Go to [**tutorial-indicators notebook**](tutorial-indicators.ipynb#Access-and-analyze-pre-calculated-climate-indicators-for-an-ensemble-of-climate-simulations) and re-run **Steps 1 to 7**, but update the **Step 3** inputs as follows:
- `ssps = ['ssp370', 'ssp245']`
- `spatial_avg = True`
- `time_avg = False`
- `ensemble_percentiles = True`

3. Running the workflow with these settings will create a new dataset containing:
- seasonal values,
- spatial averages over each sub-region,
- no temporal averaging across years,
- and ensemble percentiles for each scenario.
    
4. The new dataset will be saved as a `.zarr` file in the output folder.
- output file with pattern : *\*ensemble_percentiles_spatial-avg_*\*

5. Open this dataset and select:
- the `seasonal` temporal group,
- one season (for example, `MAM`),
- the variable `prtot`,
- and one sub-region (for example, `Central`).

6. Create a time series plot, similar to those shown on climatedata.ca, for the selected sub-region under different SSP scenarios.

Specify the following options below.  This should be changed to match the options that you used when rerunning the data processing in [**tutorial-indicators notebook**](tutorial-indicators.ipynb#Access-and-analyze-pre-calculated-climate-indicators-for-an-ensemble-of-climate-simulations)  
<div class="alert alert-success">
<strong>User input required below</strong> 
</div>


In [ ]:
# === Change only the values on the right-hand side of the `=` signs. ====
data_source = "CanDCS-M6"  # data source

# region file (shapefile or GeoJSON)
polygon_file = "GIS-Data/NS_regional_admin_boundaries/NS_regional_admin_boundaries.shp"  # Nova Scotia admin regions example
# polygon_file = "GIS-Data/Maritimes_National_Parks/Maritimes_National_Parks_2026_02_04.shp"    # Maritime National Parks example
# polygon_file = "GIS-Data/watersheds/watersheds_southernQC.shp"                                # Quebec example

# === Change only the values on the right-hand side of the `=` signs. ====
temp_group = (
    "seasonal"  # which dataset to access: one of "seasonal", "monthly", "annual" etc.
)

# Where to save Zarr and CSV files?
output_folder = Path("output/" + Path(polygon_file).stem)

From the options used we should be able to find the appropriate zarr output file and load it 

In [ ]:
# find appropriate output data
zarr = list(
    output_folder.joinpath(data_source, "zarr").glob(
        f"{temp_group}_*ensemble_percentiles_spatial-avg_*.zarr"
    )
)
ds1 = None
if len(zarr) != 1:
    print(
        f"found {len(zarr)} datasets - {zarr} .. expected one. Make sure to generate a new output with user specifications described above."
    )
else:
    # if the dataset exists, open it and subset to the selected variable, season and sub-region
    zarr = zarr[0]
    ds1 = xr.open_zarr(zarr, decode_timedelta=False)
    print(f"Successfully found {zarr.name} : loading dataset ... \n")

print(f"Available variables : {', '.join([s for s in ds1.data_vars])}\n")
print(f"Available seasons : {', '.join([s for s in ds1.temp_grp.values])}\n")
print(f"Potential 'reg_field' options and associated values... ")
for coord in [c for c in ds1.coords if ds1[c].dims == ('geom',) and c not in ['geom', 'lat','lon','objectid', 'shape_area', 'shape_leng']]:
    print(f'"{coord}" : {[c for c in ds1[coord].values]}')

Specify season, variable and sub-region to plot. 
<div class="alert alert-success">
<strong>User input required below</strong> 
</div>


In [ ]:
analyze_variable = "prtot"  # Which variable to plot. Must be one of the 'Available variables'  from the previous cell's print out
season = "MAM"  # Which season to plot. Must be one of the 'Available seasons'  from the previous cell's print out
reg_field = "region"  # Coordinate with sub-region names: Choose the left hand side of the appropriate print out

# Choose from one of the available sub-regions
analyze_region = "Central"  # Choose from the right hand side of the print out for your 'reg_field' choice (e.g. one of "Central", "Eastern", "Northern", "Western")
# analyze_region = "CAPE BRETON HIGHLANDS NATIONAL PARK OF CANADA" # for Maritime National Parks example
# analyze_region = "Central Ottawa - Dumoine"                      # for QC watersheds example

Run the cell below to create the time series plot and save it. In the plot, the solid lines show the ensemble median, and the shaded envelopes show the 10th–90th percentile range across the ensemble.

In [ ]:
# import the plotting library
import figanos.matplotlib as fg

ds1 = xr.open_zarr(zarr, decode_timedelta=False)
ds1 = ds1.sel(geom=ds1[reg_field] == analyze_region, temp_grp=season)
ds1 = ds1.drop_vars([v for v in ds1.data_vars if v != analyze_variable])
ds1 = ds1.sortby("scenario")  # order the scenario plots

# create a dictionary with keys as scenario names and values as the corresponding data arrays for the selected variable, season and sub-region. This will be used for plotting.
data= {}
for scen in ds1.scenario:
    data[f"{scen.values}"] = (
        ds1[analyze_variable].sel(scenario=scen).squeeze()
    )

# plot the time series using the figanos library
f, ax = plt.subplots(figsize=(15, 5))

# for details on fg.timeseries, see: https://figanos.readthedocs.io/en/latest/notebooks/figanos_timeseries.html
fg.timeseries(ax=ax, data=data, legend="lines", show_lat_lon=False)
tit1 = f"{ds1[analyze_variable].attrs['long_name'].rstrip('.').capitalize()} - {analyze_region} : {season}"
ax.set_title(tit1, loc="left")
ax.legend(loc="upper left")

# save the figure
fig_folder = Path("figures")
fig_folder.mkdir(exist_ok=True)  # create folder if it doesn't exist
fig_path = (
    fig_folder / f"time_series_{analyze_region}_{analyze_variable}_{season}.png"
)
plt.savefig(fig_path)
print(f"Figure saved to {fig_path}")

plt.show()

### **Example 3 - Choropleth maps: Add indicator values to the original polygon layer**

This example compares the ensemble median winter (DJF) total precipitation (`prtot`) for the baseline period (1991–2020) and future period (2071–2100) under SSP3-7.0 using interactive choropleth maps.
A choropleth map is a map in which each geographic region is shaded according to the value of a variable, making it easy to visualize spatial patterns across polygons.

To generate the maps:

1. Go to [**tutorial-indicators notebook**](tutorial-indicators.ipynb#Access-and-analyze-pre-calculated-climate-indicators-for-an-ensemble-of-climate-simulations) and re-run **Steps 1 to 7** but update the **Step 3** inputs as follows:
- `spatial_avg = True`
- `time_avg = True`
- `ensemble_percentiles = True`

This will generate and save a dataset of **spatially averaged, 30-year mean values with ensemble percentiles**.

2. Open the newly generated output dataset (`*ensemble_percentiles_30yAvg_spatial-avg*.zarr`) and use it to:
* Add data columns to the original polygon layer
* Export the updated polygon layer to **GeoJSON** (or shapefile)
* Create interactive maps in the PAVICS environment

Specify the following options below.  This should be changed to match the options that you used when rerunning the data processing in [**tutorial-indicators notebook**](tutorial-indicators.ipynb#Access-and-analyze-pre-calculated-climate-indicators-for-an-ensemble-of-climate-simulations)  
<div class="alert alert-success">
<strong>User input required below</strong> 
</div>


In [ ]:
# === Change only the values on the right-hand side of the `=` signs. ====
data_source = "CanDCS-M6"  # data source

# region file (shapefile or GeoJSON)
polygon_file = "GIS-Data/NS_regional_admin_boundaries/NS_regional_admin_boundaries.shp"  # Nova Scotia admin regions example
# polygon_file = "GIS-Data/Maritimes_National_Parks/Maritimes_National_Parks_2026_02_04.shp"    # Maritime National Parks example
# polygon_file = "GIS-Data/watersheds/watersheds_southernQC.shp"                                # Quebec example

# === Change only the values on the right-hand side of the `=` signs. ====
temp_group = (
    "seasonal"  # which dataset to access: one of "seasonal", "monthly", "annual" etc.
)

# Where to save Zarr and CSV files?
output_folder = Path("output/" + Path(polygon_file).stem)

From the options used we should be able to find the appropriate zarr output file and load it 

In [ ]:
# find appropriate output data
zarr = list(
    output_folder.joinpath(data_source, "zarr").glob(
        f"{temp_group}_*ensemble_percentiles_30yAvg_spatial-avg_*.zarr"
    )
)
ds1 = None
if len(zarr) != 1:
    print(
        f"found {len(zarr)} datasets - {zarr} .. expected one. Make sure to generate a new output with user specifications described above."
    )
else:
    # if the dataset exists, open it and subset to the selected variable, season and sub-region
    zarr = zarr[0]
    ds1 = xr.open_zarr(zarr, decode_timedelta=False)
    print(f"Successfully found {zarr.name} : loading dataset ... \n")



Add climate data attribute columns to our original shapefile

In [ ]:
# export only the ensemble median : could add [10, 50, 90]
ds1 = xr.open_zarr(zarr, decode_timedelta=False)
keep_perc = [50]

# load the original 'empty' polygon layer
gdf = gpd.read_file(polygon_file).to_crs(epsg=4326)
org_columns = gdf.columns
   
# define output shp file path
outpoly = output_folder.joinpath(
    data_source,
    "geojson",
    f"{zarr.stem}.geojson",
)

# create parent directory
outpoly.parent.mkdir(exist_ok=True, parents=True)

# concatenate climate data output as new shapefile columns
for vv in ds1.data_vars:
    for ssp in ds1.scenario.values:
        for seas in ds1.temp_grp.values:
            for hh in ds1.horizon.values:
                for perc in keep_perc:
                    # define column name
                    vvv = f"{vv}-{seas}-{hh}-{ssp}_p{perc}"
                    # get data values
                    data = (
                        ds1[vv]
                        .sel(
                            temp_grp=seas,
                            scenario=ssp,
                            percentiles=perc,
                            time=(ds1.horizon == hh),
                        )
                        .squeeze()
                    )
                    # get units as well
                    units = data.attrs["units"]
                    # convert to pandas df
                    data = data.to_dataframe()
                    # append new data column to polygon layer
                    gdf[f"{vvv} ({units})"] = data[vv]

# export to geojson
gdf.to_file(outpoly, driver="GeoJSON")

print(f"Updated polygon layer exported to: {outpoly}")

Let's take a look at the output. The output is a GeoJSON file with the same geometry as the input polygon file, but with additional columns for each variable, season, horizon, scenario, and percentile combination.

In [ ]:
print(outpoly)
gdf = gpd.read_file(outpoly)
display(gdf.head())
print(f"Available data columns for chloropleth maps : {'\n'.join([c for c in gdf.columns if c not in org_columns])}")

Run the cell below to plot the ensemble median of average winter total precipitation for sub-regions for baseline (1991–2020) and future (2071–2100) periods under SSP3-7.0. By comparing the two maps, you can see how precipitation levels vary across regions and how they are projected to change over time.

In [ ]:
# variable for baseline period
var1 = f"prtot-DJF-1991-2020-ssp370_p50 (mm)" # choose from list in print out above; You may need to scroll to see all options

# variable for future period; 
var2 = var1.replace('1991-2020', '2071-2100') # use 'replace' method on the 'horizon' portion of our baseline choice; ensures we are always comparing the same season and variable 

# set color limits (min and max of our two variables)
clim = (gdf[[var1, var2]].min().min(), gdf[[var1, var2]].max().max())

# plot interactive map
gdf.hvplot(
    c=var1,
    clim=clim,
    geo=True,
    tiles="CartoLight",
    alpha=0.65,
    title=var1,
    cmap="spectral",
) + gdf.hvplot(
    c=var2,
    clim=clim,
    geo=True,
    tiles="CartoLight",
    alpha=0.65,
    title=var2,
    cmap="spectral",
)

### **Example 4 - Maps of gridded data for our region** 
#### **4.1 Interactive maps on PAVICS**

This example shows how to visualize pre-calculated indicators as gridded maps for a selected region, compare a reference period with other time horizons, and export the results as GeoTIFF files for use in GIS.

To generate the maps:

1. Go to [**tutorial-indicators notebook**](tutorial-indicators.ipynb#Access-and-analyze-pre-calculated-climate-indicators-for-an-ensemble-of-climate-simulations) and re-run **Steps 1 to 7** but update the **Step 3** inputs as follows:
- `spatial_avg = False`
- `time_avg = True`
- `ensemble_percentiles = True`

    This will generate and save a dataset of **gridded 30-year mean values with ensemble percentiles**.

2. Open the newly generated output dataset (`*ensemble_percentiles_30yAvg.zarr`) and use it to:
* Create interactive plots in the PAVICS environment
* Create a simple time-slider widget to navigate across horizons
* Export gridded outputs to GeoTIFF for GIS users

Specify the following options below.  This should be changed to match the options that you used when rerunning the data processing in [**tutorial-indicators notebook**](tutorial-indicators.ipynb#Access-and-analyze-pre-calculated-climate-indicators-for-an-ensemble-of-climate-simulations)  
<div class="alert alert-success">
<strong>User input required below</strong> 
</div>


In [ ]:
# === Change only the values on the right-hand side of the `=` signs. ====
data_source = "CanDCS-M6"  # data source

# region file (shapefile or GeoJSON)
polygon_file = "GIS-Data/NS_regional_admin_boundaries/NS_regional_admin_boundaries.shp"  # Nova Scotia admin regions example
# polygon_file = "GIS-Data/Maritimes_National_Parks/Maritimes_National_Parks_2026_02_04.shp"    # Maritime National Parks example
# polygon_file = "GIS-Data/watersheds/watersheds_southernQC.shp"                                # Quebec example

# === Change only the values on the right-hand side of the `=` signs. ====
temp_group = (
    "seasonal"  # which dataset to access: one of "seasonal", "monthly", "annual" etc.
)

# Where to save Zarr and CSV files?
output_folder = Path("output/" + Path(polygon_file).stem)

From the options used we should be able to find the appropriate zarr output file and load it 

In [ ]:
# find appropriate output data
zarr = list(
    output_folder.joinpath(data_source, "zarr").glob(
        f"{temp_group}_*ensemble_percentiles_30yAvg.zarr"
    )
)
ds1 = None
if len(zarr) != 1:
    print(
        f"found {len(zarr)} datasets - {zarr} .. expected one. Make sure to generate a new output with user specifications described above."
    )
else:
    # if the dataset exists, open it and subset to the selected variable, season and sub-region
    zarr = zarr[0]
    ds1 = xr.open_zarr(zarr, decode_timedelta=False)
    print(f"Successfully found {zarr.name} : loading dataset ... \n")

print(f"Available emissions scenarios : {', '.join([s for s in ds1.scenario.values])}\n")
print(f"Available variables : {', '.join([s for s in ds1.data_vars])}\n")
print(f"Available seasons : {', '.join([s for s in ds1.temp_grp.values])}\n")
print(f"Available horizons : {', '.join([s for s in ds1.horizon.values])}\n")

Specify scenario, variable, season and reference period to use in the plot. 
<div class="alert alert-success">
<strong>User input required below</strong> 
</div>


In [ ]:
scenario = "ssp370" # Which SSP to plot. Must be one of the 'Available emissions scenarios' from the previous cell's print out
analyze_variable = "prtot"  # Which variable to plot. Must be one of the 'Available variables' from the previous cell's print out
season = "MAM"  # Which season to plot. Must be one of the 'Available seasons' from the previous cell's print out
ref_period = "1991-2020" # Which reference period or horizon to use for comparisons.  Must be one of the 'Available horizons' from the previous cell's print out

Run the cell below to create the interactive maps with a slider to explore other horizons.

In [ ]:
dsall = xr.open_zarr(zarr, decode_timedelta=False)
ds1 = dsall.sel(
    scenario=scenario, temp_grp=season, percentiles=keep_perc
).squeeze()

xlim = [ds1.lon.min().values - 0.5, ds1.lon.max().values + 0.5]
ylim = [ds1.lat.min().values - 0.5, ds1.lat.max().values + 0.5]
# define color limits for all time periods
clim = (
    ds1[analyze_variable].min().values,
    ds1[analyze_variable].max().values,
)

# define common keyword args for both reference and future maps
kwargs = dict(
    xlim=xlim,
    ylim=ylim,
    clim=clim,
    geo=True,
    tiles="CartoLight",
    alpha=0.65,
    cmap="spectral",
    attr_labels=False,
    framewise=False,
)

# reference data plot
ref_plot = (
    ds1[analyze_variable]
    .sel(time=ds1.horizon == ref_period)
    .squeeze()
    .hvplot.image(title=ref_period, **kwargs)
)

# create a time-slider with dataset horizons as options
time_opt_dict = {
    f"{ds1.sel(time=tt).horizon.values}": tt for tt in ds1.time.values
}
time_slider = pn.widgets.DiscreteSlider(
    name="Time", value=ds1.time.values[-1], options=time_opt_dict
)

# future selection bound to time slider widget
fut_plot = (
    ds1[analyze_variable]
    .interactive(loc="bottom")
    .sel(time=time_slider)
    .hvplot.image(**kwargs)
)

# plot side by side (+)
plt1 = ref_plot + fut_plot

# create a title using metadata and season choice
md_title = pn.pane.Markdown(
    f"## {ds1[analyze_variable].attrs['long_name'].rstrip('.').capitalize()} ({ds1[analyze_variable].attrs['units']})<br>Season : {season}<br>Emissions : {scenario.upper()}"
)

# display title and plot in the jupyterlab
display(pn.Column(md_title, plt1))

#### **4.2 Exporting gridded outputs to GeoTIFF**

The code below loops over all non-spatial dimensions in the dataset and exports each 2D slice as a separate GeoTIFF file. These files can then be opened in standard GIS software.

To download them from the PAVICS environment, right-click the `geotiff` subfolder in the output directory and select "Download as an archive" to quickly save all layers locally.

In [ ]:
# loop over data variables
for vv in dsall.data_vars:
    print(f"Exporting {vv} to geotiffs ... ")
    # loop over non-spatial dimensions
    for tt in dsall.time:
        for pp in keep_perc:
            for ss in dsall.scenario:
                for seas in dsall.temp_grp:
                    # select 2D data slice
                    data = dsall[vv].sel(
                        time=tt, percentiles=pp, scenario=ss, temp_grp=seas
                    )  # .squeeze()

                    # create output file path
                    outtiff = Path(
                        output_folder
                        / data_source
                        / "geotiff"
                        / vv
                        / f"{zarr.stem}_{ss.values}_{vv}({data.attrs['units']})_{data.horizon.values}_{seas.values}_p{pp}.tiff"
                    )

                    # creat parent folder
                    outtiff.parent.mkdir(parents=True, exist_ok=True)

                    # Geotiff anchor point is top-left corner
                    # sort data w/ lat ascending=False
                    # sort data w/ lon ascending=True
                    data = (
                        data.sortby(["lat"], ascending=False)
                        .sortby(["lon"], ascending=True)
                        .squeeze()
                        .rio.set_crs("epsg:4326")
                        .rio.set_spatial_dims(x_dim="lon", y_dim="lat")
                    )

                    # write data to file
                    data.rio.to_raster(outtiff)

    print("Done. Files saved to: " + str(outtiff.parent))

### **Example 5 - Climate Stripes visualization**
This example shows how to create a climate stripes visualization for a region of interest. Climate stripes figures are simple color‑bar visuals that show, for example, how temperatures evolve over time. They’re useful in climate‑change projection work when you want to quickly communicate long‑term trends without technical details.

To generate the climate stripes figure:

1. Go to [**tutorial-indicators notebook**](tutorial-indicators.ipynb#Access-and-analyze-pre-calculated-climate-indicators-for-an-ensemble-of-climate-simulations) and re-run **Steps 1 to 7** but update the **Step 3** inputs as follows:
- `spatial_avg = True`
- `time_avg = False`
- `ensemble_percentiles = True`
- `ssps = ["ssp126", "ssp245", "ssp370", "ssp585"]`
- `variables = {"annual":["tg_mean"]}`

This will generate and save a dataset with **spatially averaged, annual values and ensemble percentiles** calculated.

2. Open the newly generated output dataset (`*ensemble_percentiles_spatial-avg_*.zarr`) and use it to:
* Plot a climate stripes figure
* Export to png


Specify the following options below.  This should be changed to match the options that you used when rerunning the data processing in [**tutorial-indicators notebook**](tutorial-indicators.ipynb#Access-and-analyze-pre-calculated-climate-indicators-for-an-ensemble-of-climate-simulations)  
<div class="alert alert-success">
<strong>User input required below</strong> 
</div>


In [ ]:
# === Change only the values on the right-hand side of the `=` signs. ====
data_source = "CanDCS-M6"  # data source

# region file (shapefile or GeoJSON)
polygon_file = "GIS-Data/NS_regional_admin_boundaries/NS_regional_admin_boundaries.shp"  # Nova Scotia admin regions example
# polygon_file = "GIS-Data/Maritimes_National_Parks/Maritimes_National_Parks_2026_02_04.shp"    # Maritime National Parks example
# polygon_file = "GIS-Data/watersheds/watersheds_southernQC.shp"                                # Quebec example

# === Change only the values on the right-hand side of the `=` signs. ====
temp_group = (
    "annual"  # which dataset to access: one of "seasonal", "monthly", "annual" etc
)

# Where to save Zarr and CSV files?
output_folder = Path("output/" + Path(polygon_file).stem)

From the options used we should be able to find the appropriate zarr output file and load it 

In [ ]:
# find appropriate output data
zarr = list(
    output_folder.joinpath(data_source, "zarr").glob(
        f"{temp_group}_*ensemble_percentiles_spatial-avg_*.zarr"
    )
)
ds1 = None
if len(zarr) != 1:
    print(
        f"found {len(zarr)} datasets - {zarr} .. expected one. Make sure to generate a new output with user specifications described above."
    )
else:
    # if the dataset exists, open it and subset to the selected variable, season and sub-region
    zarr = zarr[0]
    ds1 = xr.open_zarr(zarr, decode_timedelta=False)
    print(f"Successfully found {zarr.name} : loading dataset ... \n")
    
print(f"Available emissions scenarios : {', '.join([s for s in ds1.scenario.values])}\n")
print(f"Available variables : {', '.join([s for s in ds1.data_vars])}\n")
print(f"Available seasons : {', '.join([s for s in ds1.temp_grp.values])}\n")
print(f"Potential 'reg_field' options and associated values... ")
for coord in [c for c in ds1.coords if ds1[c].dims == ('geom',) and c not in ['geom', 'lat','lon','objectid', 'shape_area', 'shape_leng']]:
    print(f'"{coord}" : {[c for c in ds1[coord].values]}')

Specify scenarios, variable, variable and sub-region to plot. 
<div class="alert alert-success">
<strong>User input required below</strong> 
</div>


In [ ]:
scenario = ["ssp126", "ssp245", "ssp370", "ssp585"] # Which SSP to plot. Must be one (or more) of the 'Available emissions scenarios' from the previous cell's print out
analyze_variable = "tg_mean"  # Which variable to plot. Must be one of the 'Available variables' from the previous cell's print out
season = "annual"  # Which season to plot. Must be one of the 'Available seasons' from the previous cell's print out

reg_field = "region"  # Coordinate with sub-region names: Choose the left hand side of the appropriate print out
# Choose from one of the available sub-regions
analyze_region = "Central" # Choose from the right hand side of the print out for your 'reg_field' choice (e.g. one of "Central", "Eastern", "Northern", "Western")

plot_perc = [50]  # plot the ensemble median for each ssp

color_bar_label = f'Mean annual mean temperature (°C) \n {plot_perc[0]}th percentile of {data_source}' # provide a custom label for the colobar

divide_year = 2015 # Year at which the plot is divided into scenarios. For CMIP6 2015 is the year between historical and ssp emissions

cmap = 'temp_seq' # colormap to apply; See Figanos documentation for available options : https://figanos.readthedocs.io/en/latest/notebooks/figanos_maps.html#

Run the cell below to create the figure.

In [ ]:
import figanos.matplotlib as fg
ds1 = xr.open_zarr(zarr, decode_timedelta=False)
# extract data for plotting
data1 = ds1.sel(geom=ds1[reg_field] == analyze_region, temp_grp=season, percentiles=plot_perc)[analyze_variable]
# determine colormap center value
cmap_center = data1.mean().load()
# create a dictionary : one key per ssp with only that data
data_dict = {f"{s.values}": data1.sel(scenario=s).squeeze() for s in data1.scenario}
# plot the data 
# add division in 2015 : transition year between historical and ssp runs 
f = fg.stripes(data_dict, cmap_center=cmap_center, cmap=cmap, divide=divide_year, cbar_kw={'label':color_bar_label})
# plot the data
t = plt.title(f"{analyze_region} : Climate stripes visual", loc='left')

# save the figure
fig_folder = Path("figures")
fig_folder.mkdir(exist_ok=True)  # create folder if it doesn't exist
fig_path = (
    fig_folder / f"climate_stripes_{analyze_region}_{analyze_variable}_{season}.png"
)
plt.savefig(fig_path)
plt.show()
print(f"Figure saved to {fig_path}")